# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ranamohsincodes/flyranktask1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet') LIMIT 1").df()
print(schema)

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page's daily performance record for one client, on one date (grain: client_hash_id + content_hash_id + report_date). Time window for this analysis: March 2026 (a mid-panel month, not the final sealed month).

In [7]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print("Duplicate (client, content, date) rows found:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, date) rows found: 0


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ranamohsincodes/flyranktask1"
REPO_DIR = "flyranktask1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

!pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
print("Connected. Ready to query.")


Working dir: /content/flyranktask1/flyranktask1/flyranktask1
Connected. Ready to query.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields (predictive signals, knowable at decision time):
- gsc_impressions, gsc_clicks, gsc_avg_position — daily Search Console performance.
- ga4_sessions, ga4_engaged_sessions, ga4_total_engagement_sec — daily engagement signals.

Label field (what I'm predicting/ranking):
- A trend proxy I build myself by comparing gsc_impressions/gsc_clicks across days within the month — not a pre-existing label column in the warehouse.

Context fields (identifiers, not model inputs):
- client_hash_id, content_hash_id, report_date — used for joining, grouping, and filtering only.

Excluded fields (deliberately not used):
- gsc_data_available and ga4_data_available are used only as filters (to keep valid rows), never as predictive features, since they describe data quality rather than content performance.
- GA4 session-source breakdowns (sessions_organic, sessions_paid, sessions_ai, etc.) are excluded from this pass to keep the feature set small and focused.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries below confirm: (1) the grain is correct (already shown in Section 1), (2) the row count and date span of my slice, (3) availability — how many rows survive an IS TRUE filter on real GSC data.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Row count and date span
q2 = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as start_date, MAX(report_date) as end_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("Row count and date span:")
print(q2)

# Availability, filtered with IS TRUE
q3 = con.sql(f"""
    SELECT COUNT(*) as available_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()
print("Rows with GSC data available:")
print(q3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Row count and date span:
   total_rows start_date   end_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with GSC data available:
   available_rows
0         3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell you:

1. Unbalanced history — clients joined at different times, so some have months of history while others have only a few weeks. Comparing "decline" across clients isn't a fair apples-to-apples comparison.

2. GSC-only early rows — earlier records may only have Search Console data (gsc_*) without GA4 engagement data, since ga4_data_available can be false while gsc_data_available is true. Any feature relying on GA4 fields will silently drop or bias toward clients who connected GA4 later.

3. Window overlaps — a single month's snapshot can't separate a real multi-week decline from short-term noise (e.g. a single bad week dragging down a monthly average). This data shows what happened, not durable cause-and-effect.

4. This data cannot explain WHY a page's performance changed — no information on algorithm updates, competitor changes, or content edits is included, so any trend I find is observed, not causally explained.

The trap: adding a label-derived column on purpose to show how leakage creates a fake "perfect" result.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.# Build a small feature frame from March 2026
# Build a small feature frame from March 2026
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_avg_position
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    LIMIT 2000
""").df()

# Define a simple label: 1 if clicks are above the month's median, else 0
median_clicks = feat["gsc_clicks"].median()
feat["label_high_clicks"] = (feat["gsc_clicks"] > median_clicks).astype(int)

# THE TRAP: deliberately add a column derived directly from the label
feat["leaky_column"] = feat["gsc_clicks"] * 2  # literally built from the same column as the label

# "Quick score": correlation between the leaky feature and the label
leaky_corr = feat["leaky_column"].corr(feat["label_high_clicks"])
print(f"Quick score WITH leaky column: {leaky_corr:.3f}  <- suspiciously high, this is the trap")

# Now delete the leaky column and keep the honest feature set
feat = feat.drop(columns=["leaky_column"])
honest_corr = feat["gsc_avg_position"].corr(feat["label_high_clicks"])
print(f"Honest score WITHOUT leaky column (using gsc_avg_position instead): {honest_corr:.3f}")
print("Leaky column removed. Honest columns:", feat.columns.tolist())


Quick score WITH leaky column: 0.646  <- suspiciously high, this is the trap
Honest score WITHOUT leaky column (using gsc_avg_position instead): -0.165
Leaky column removed. Honest columns: ['client_hash_id', 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'label_high_clicks']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.